# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the [FAIR^2 Mass Spectrometry dataset](https://doi.org/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library and the Croissant schema specification.

### Dataset Source
The dataset is described using a Croissant schema accessible via the following URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Install mlcroissant
!pip install mlcroissant

## 1. Data Loading
Load the dataset schema and inspect high-level metadata using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import warnings
warnings.filterwarnings("ignore")

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Print the dataset metadata
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Authors: {getattr(metadata, 'author', 'n/a')}")
print(f"Version: {getattr(metadata, 'version', 'n/a')}")
print(f"Date Published: {getattr(metadata, 'datePublished', 'n/a')}")
print(f"License: {getattr(metadata, 'license', 'n/a')}")

## 2. Data Overview
Explore the available record sets, and their fields, using their `@id` values. This helps identify what record sets and fields are available for extraction.

The Croissant schema may contain several record sets. Their identifiers (`@id`) are used throughout this notebook to reference and extract data.

In [ ]:
# List all record sets and details
record_set_ids = []
for record_set in dataset.record_sets:
    print(f"Record Set: {record_set.name}")
    print(f"  @id: {record_set.id}")
    print(f"  Description: {getattr(record_set, 'description', 'No description')}")
    print(f"  Fields:")
    for field in record_set.fields:
        print(f"    - {field.name} (@id: {field.id}, type: {getattr(field, 'data_type', 'n/a')})")
    print()
    record_set_ids.append(record_set.id)

print(f"Found Record Sets: {record_set_ids}")

## 3. Data Extraction
Extract records from each record set and load them into Pandas DataFrames for analysis.

We use the `@id` of each record set for mlcroissant extraction. This approach is flexible and robust, as it ensures the correct record sets are loaded regardless of internal structure.

In [ ]:
# Extract all records into DataFrames for each record set
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records for record set {record_set_id} ...")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  Loaded {len(df)} records. Columns: {df.columns.tolist()}")
    else:
        print(f"  No records found.")

# For demonstration, pick the first record set with data
main_record_set = next(iter(dataframes)) if dataframes else None
if main_record_set:
    print(f"\nData Preview for main record set (@id = {main_record_set}):\n")
    display(dataframes[main_record_set].head())
else:
    print("No record set with records found.")

## 4. Exploratory Data Analysis (EDA)
Analyze numerical fields, perform data normalization, outlier filtering, and grouping. The following demonstrates choosing a numerical field for this analysis based on available columns.

All field references use their column `@id` according to the Croissant schema.

In [ ]:
if main_record_set is not None:
    df = dataframes[main_record_set]

    # Identify numeric columns (float/int)
    numeric_columns = df.select_dtypes(include=['number']).columns.tolist()
    print(f"Numeric columns found: {numeric_columns}")

    # Choose a numeric field to analyze (default: first one if available)
    numeric_field = numeric_columns[0] if numeric_columns else None

    if numeric_field:
        # Example threshold for filtering
        threshold = df[numeric_field].quantile(0.75)
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records where '{numeric_field}' > {threshold}\n")
        display(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized '{numeric_field}':")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # If a categorical field is present, group by this field
        # Try to pick a non-numeric field
        candidate_group_fields = [col for col in df.columns if col not in numeric_columns]
        group_field = candidate_group_fields[0] if candidate_group_fields else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nMean of '{numeric_field}' grouped by '{group_field}':")
            display(grouped_df.head())
        else:
            print("No suitable non-numeric group field found.")
    else:
        print("No numeric fields available for EDA.")
else:
    print("No data available for EDA.")

## 5. Visualization
Let's visualize the distribution of a quantitative field and relationships in the data using matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set is not None and numeric_field:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # If we also found a group field, show boxplot
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(12, 5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
- We loaded the Mass Spectrometry dataset using the Croissant schema and explored its record sets using only their `@id`s.
- Data were extracted with `mlcroissant` into pandas DataFrames. We identified available numeric fields, performed basic filtering, normalization, and grouped analysis by other attributes.
- Visualizations showed the distribution of selected numeric fields and summary by categories.

This workflow demonstrates how to access and process complicated, semantically-rich scientific data packages using `mlcroissant`, providing a strong foundation for further domain-specific analyses.